# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sai875/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

**Lane:** Lane 2 — Refresh / Content Opportunity Scoring

This notebook defines and checks a small, auditable data contract on the March 2026 warehouse partition. It builds a feature frame from information available at the decision date, then demonstrates how a future-window field leaks the label. The June 2026 `_sample` partition is reserved as a sealed test period and is not used here.

**Access requirement:** request and accept access to [FlyRank/internship-warehouse](https://huggingface.co/datasets/FlyRank/internship-warehouse), then store a READ token as a Colab Secret named `HF_TOKEN`. Do not paste a token into notebook code or outputs.

In [1]:
import os
import sys
import subprocess
from pathlib import Path

# Open from this repository's Colab badge, or use the checked-out repository in CI/local Jupyter.
REPO_URL = "https://github.com/Sai875/flyrank-ml-internship.git"
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_DIR = Path("/content/flyrank-ml-internship")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "work/notebooks/w03_data_contract.ipynb").exists():
            os.chdir(candidate)
            break

%pip -q install duckdb pandas numpy scikit-learn

import duckdb
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

token = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN") or token
except Exception:
    # Outside Colab, the optional GitHub Actions secret is supplied via environment.
    pass
if not token:
    raise RuntimeError(
        "HF_TOKEN is missing. In Hugging Face, accept the FlyRank/internship-warehouse "
        "access terms first, then add a READ token in Colab Secrets as HF_TOKEN."
    )

con = duckdb.connect()
safe_token = token.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

MONTH = "month=2026-03"
FACT = (
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/"
    f"fact_content_daily_performance/{MONTH}/*.parquet')"
)
DIM_CONTENT = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')"
DIM_CLIENTS = "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')"
print("Connected. Working directory:", os.getcwd())


Note: you may need to restart the kernel to use updated packages.


Connected. Working directory: /home/runner/work/flyrank-ml-internship/flyrank-ml-internship


## 1. Data contract — five plain-English answers

| Contract item | Definition |
|---|---|
| **Row grain** | One content item for one reporting date (a content-day), identified by `client_hash_id`, `content_hash_id`, and `report_date`. |
| **Source tables** | `fact_content_daily_performance` for daily search/engagement observations; `dim_content` and `dim_clients` for approved descriptive context. |
| **Time window** | March 1–31, 2026, a mid-panel development month. The June 2026 `_sample` partition remains sealed and is not queried. |
| **Target / proxy** | An observed click-decline proxy: future 7-day clicks are at least 20% below the preceding 7-day clicks, for eligible content-days with nonzero prior clicks. This is a retrospective label, not a claim about Google's ranking system. |
| **Exclusions** | Pseudonymous IDs and dates are used for joins/grouped splitting only. Future-window clicks, target fields, and any label-derived fields are excluded from final model inputs. GA4 values are not treated as real when `ga4_data_available IS TRUE` is false or unknown; raw zero position is treated as missing. |

The model features below use only data available on or before decision day `d`. The proxy label intentionally looks forward; that future information must never be an input.

## 2. Verification query A — grain

A correct content-day grain should not contain duplicate `(client_hash_id, content_hash_id, report_date)` keys. An empty result passes this check.

In [2]:
# Verification query A: duplicate content-day keys (expected: zero rows).
grain_probe = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS row_count
    FROM {FACT}
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 10
""").df()
display(grain_probe)
assert grain_probe.empty, "Duplicate content-day keys found; revisit the stated grain."
print("PASS: no duplicate content-day keys in the probe.")


,client_hash_id,content_hash_id,report_date,row_count


PASS: no duplicate content-day keys in the probe.


## 3. Verification query B — row count and date span

This reports the observed row count, distinct content/client counts, and minimum/maximum dates for the selected March partition.

In [3]:
# Verification query B: counts and date span.
slice_shape = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        COUNT(DISTINCT content_hash_id) AS n_content_items,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM {FACT}
""").df()
display(slice_shape)
assert int(slice_shape.loc[0, "n_rows"]) > 0, "The selected partition returned no rows."
assert str(slice_shape.loc[0, "min_date"])[:10] == "2026-03-01", "Unexpected minimum date."
assert str(slice_shape.loc[0, "max_date"])[:10] == "2026-03-31", "Unexpected maximum date."


,n_rows,n_content_items,n_clients,min_date,max_date
0,9841378,331437,55,2026-03-01,2026-03-31


## 4. Verification query C — availability uses `IS TRUE`

A value that is false or unknown is not evidence of data availability. This query counts only rows where `ga4_data_available IS TRUE`; it does not infer availability from zeros.

In [4]:
# Verification query C: availability must be explicitly TRUE.
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(
            100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)
            / NULLIF(COUNT(*), 0),
            1
        ) AS pct_available
    FROM {FACT}
""").df()
display(availability)
assert int(availability.loc[0, "total_rows"]) > 0


,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


## 5. Build a feature frame with no more than five predictive features

The frame is limited to a deterministic sample of content items to keep the notebook practical in Colab. Rolling features are calculated per content item. The target compares the 7 days after decision day `d` with the 7 days before it; decision dates are restricted to March 8–24 so both windows remain inside March.

| Feature | Definition | Available when? |
|---|---|---|
| `clicks_7d` | GSC clicks from `d-6` through `d` | At decision day `d` |
| `impressions_7d` | GSC impressions from `d-6` through `d` | At decision day `d` |
| `position_7d` | Mean daily weighted position over `d-6` through `d`; zero/no-position values become missing | At decision day `d`; missing if no valid position exists |
| `gsc_available_7d` | Whether GSC availability was explicitly true in the trailing window | At decision day `d` |

The query below uses future clicks only to construct the label and leakage demonstration. `future_clicks_7d` is not part of `FEATURES`.

In [5]:
feature_sql = f"""
WITH daily AS (
    SELECT
        client_hash_id,
        content_hash_id,
        CAST(report_date AS DATE) AS report_date,
        COALESCE(gsc_clicks, 0)::DOUBLE AS clicks,
        COALESCE(gsc_impressions, 0)::DOUBLE AS impressions,
        CASE
            WHEN COALESCE(gsc_impressions, 0) > 0
                 AND COALESCE(gsc_sum_position, 0) > 0
            THEN CAST(gsc_sum_position AS DOUBLE) / gsc_impressions
            ELSE NULL
        END AS weighted_position,
        (gsc_data_available IS TRUE) AS gsc_available
    FROM {FACT}
    WHERE report_date >= DATE '2026-03-01'
      AND report_date <= DATE '2026-03-31'
),
rolling AS (
    SELECT *,
        SUM(clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
            RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
        ) AS clicks_7d,
        SUM(impressions) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
            RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
        ) AS impressions_7d,
        AVG(weighted_position) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
            RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
        ) AS position_7d,
        MAX(CASE WHEN gsc_available THEN 1 ELSE 0 END) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
            RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
        ) AS gsc_available_7d,
        SUM(clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
            RANGE BETWEEN INTERVAL 7 DAYS PRECEDING AND INTERVAL 1 DAY PRECEDING
        ) AS prior_clicks_7d,
        SUM(clicks) OVER (
            PARTITION BY client_hash_id, content_hash_id ORDER BY report_date
            RANGE BETWEEN INTERVAL 1 DAY FOLLOWING AND INTERVAL 7 DAYS FOLLOWING
        ) AS future_clicks_7d
    FROM daily
)
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    clicks_7d,
    impressions_7d,
    position_7d,
    gsc_available_7d,
    prior_clicks_7d,
    future_clicks_7d,
    CASE
        WHEN prior_clicks_7d > 0 AND future_clicks_7d <= 0.8 * prior_clicks_7d
        THEN 1 ELSE 0
    END AS click_decline_flag
FROM rolling
WHERE report_date BETWEEN DATE '2026-03-08' AND DATE '2026-03-24'
  AND prior_clicks_7d > 0
  AND future_clicks_7d IS NOT NULL
  AND hash(content_hash_id) % 50 = 0
"""
feature_frame = con.sql(feature_sql).df()
if feature_frame.empty:
    raise RuntimeError(
        "The feature frame is empty. Check warehouse access, the March partition, "
        "and the verified column names before continuing."
    )
feature_frame["position_7d"] = feature_frame["position_7d"].replace(0, np.nan)
feature_frame["gsc_available_7d"] = feature_frame["gsc_available_7d"].astype(float)

FEATURES = ["clicks_7d", "impressions_7d", "position_7d", "gsc_available_7d"]
LABEL = "click_decline_flag"
LEAKAGE_ONLY = "future_clicks_7d"
assert len(FEATURES) <= 5
assert LEAKAGE_ONLY not in FEATURES and LABEL not in FEATURES
assert feature_frame[FEATURES].shape[1] == 4

print(f"Feature frame: {len(feature_frame):,} content-days; {feature_frame['client_hash_id'].nunique():,} client groups.")
print("Final predictive features:", FEATURES)
display(feature_frame[["report_date", LABEL] + FEATURES].head())


Feature frame: 13,219 content-days; 27 client groups.
Final predictive features: ['clicks_7d', 'impressions_7d', 'position_7d', 'gsc_available_7d']


,report_date,click_decline_flag,clicks_7d,impressions_7d,position_7d,gsc_available_7d
0,2026-03-19,0,19.0,1295.0,2.364071,1.0
1,2026-03-20,0,20.0,1416.0,2.469790,1.0
2,2026-03-21,0,20.0,1517.0,2.523975,1.0
3,2026-03-22,0,21.0,1596.0,2.617490,1.0
4,2026-03-23,0,16.0,1718.0,2.719801,1.0


## 6. Deliberate label-leakage experiment

We split by client so a client's content cannot appear in both train and test. The honest model uses only the four pre-decision features. The intentionally invalid model adds `future_clicks_7d`, a field from the label window, to demonstrate leakage. Its score is diagnostic only and must not be reported as model performance.

Precision@50 is computed on the same held-out clients. Average precision is also reported. The honest metrics are retained separately and the final feature list is asserted not to contain the leaked field.

In [6]:
def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    k = min(k, len(y_true))
    if k == 0:
        return float("nan")
    idx = np.argsort(scores)[::-1][:k]
    return float(y_true[idx].mean())

# Grouped holdout by client; no client appears on both sides.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=42)
train_idx, test_idx = next(
    splitter.split(feature_frame, feature_frame[LABEL], groups=feature_frame["client_hash_id"])
)
train = feature_frame.iloc[train_idx].copy()
test = feature_frame.iloc[test_idx].copy()
assert set(train["client_hash_id"]).isdisjoint(set(test["client_hash_id"])), "Client leakage across split."

def fit_and_score(train_df, test_df, columns):
    model = make_pipeline(
        SimpleImputer(strategy="median"),
        StandardScaler(),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    )
    model.fit(train_df[columns], train_df[LABEL].astype(int))
    scores = model.predict_proba(test_df[columns])[:, 1]
    return {
        "precision_at_50": precision_at_k(test_df[LABEL].astype(int), scores, 50),
        "average_precision": float(average_precision_score(test_df[LABEL].astype(int), scores)),
    }, model, scores

honest_metrics, honest_model, honest_scores = fit_and_score(train, test, FEATURES)
leaky_features = FEATURES + [LEAKAGE_ONLY]
leaky_metrics, leaky_model, leaky_scores = fit_and_score(train, test, leaky_features)

leakage_results = pd.DataFrame([
    {"experiment": "Honest features only", **honest_metrics},
    {"experiment": "INVALID: includes future-window clicks", **leaky_metrics},
])
display(leakage_results)
print("Honest score retained:", honest_metrics)
print("Leakage-only field excluded from final inputs:", LEAKAGE_ONLY not in FEATURES)
assert LEAKAGE_ONLY not in FEATURES
assert LABEL not in FEATURES


,experiment,precision_at_50,average_precision
0,Honest features only,0.82,0.722368
1,INVALID: includes future-window clicks,0.98,0.982367


Honest score retained: {'precision_at_50': 0.82, 'average_precision': 0.722367991198938}
Leakage-only field excluded from final inputs: True


## 7. Limitation and interpretation

**Limitation:** client history is uneven and many content-days have sparse or unavailable search data. A single March partition cannot establish that the proxy generalizes to other months, clients, or seasonal periods. The label is retrospective and the small deterministic sample is for a reproducible exercise, not a production ranking.

The leakage experiment is intentionally invalid when it includes `future_clicks_7d`. Only the honest model score belongs in any conclusion. Neither score demonstrates causation or predicts Google's ranking algorithm.

## Self-check

- [x] Five data-contract answers documented.
- [x] Exactly three verification queries: grain, row/date span, and `IS TRUE` availability.
- [x] Four predictive features, with availability timing documented.
- [x] Client-grouped holdout and deliberate label-leakage comparison included.
- [x] Future-window leakage field excluded from final model inputs; honest score retained.
- [x] One named data limitation documented.
- [ ] Run all cells in Colab with approved `HF_TOKEN`, review the actual outputs, then save the executed notebook back to this repository.